In [ ]:
import sqlite3
import pandas as pd
import unicodedata

from sqlalchemy import create_engine

engine = create_engine(
    "sqlite:///dw_acidentes.db"
)

In [ ]:
df = pd.read_csv('datatran2007.csv', sep=';',  encoding='latin1')
df

In [ ]:
df2008 = pd.read_csv('datatran2008.csv', sep=';', encoding='latin1')
df2008

In [ ]:
df2009 = pd.read_csv('datatran2009.csv', sep=';', encoding='latin1')
df2009

concatenando 3 csv de 2007 há 2009

In [ ]:
frames = [df, df2008, df2009]

df_concat = pd.concat(frames)

df_concat

exportando nova versão csv

In [ ]:
df_concat.to_csv('datatran_1.0.csv', 
                sep=';', 
                index=False, 
                encoding='latin1')

Removendo acentuação

In [ ]:
def remove_non_ascii(string: str) -> str :
    normalized = unicodedata.normalize('NFD', string)
    return normalized.encode('ascii', 'ignore').decode('latin1')

df_concat['dia_semana'] = df_concat['dia_semana'].apply(remove_non_ascii)
df_concat['municipio'] = df_concat['municipio'].apply(remove_non_ascii)
df_concat['causa_acidente'] = df_concat['causa_acidente'].apply(remove_non_ascii)
df_concat['tipo_acidente'] = df_concat['tipo_acidente'].apply(remove_non_ascii)
df_concat

In [ ]:
# Padronização para bater com dimensões

df_concat['municipio'] = df_concat['municipio'].str.strip().str.title()
df_concat['uf'] = df_concat['uf'].str.upper()
df_concat['tipo_pista'] = df_concat['tipo_pista'].str.title()

df_concat['tipo_acidente'] = df_concat['tipo_acidente'].str.strip().str.title()
df_concat['causa_acidente'] = df_concat['causa_acidente'].str.strip().str.title()

Exportando nova versão sem acentuação

In [ ]:
df_concat.to_csv('datatran_1.1.csv', 
                sep=';', 
                index=False, 
                encoding='latin1')

DataWarehouse

In [ ]:
datas = pd.date_range(start='2007-01-01', end='2009-12-31')

dim_tempo = pd.DataFrame({'data': datas})

dias_semana_pt = {
    'Monday': 'Segunda-feira',
    'Tuesday': 'Terça-feira',
    'Wednesday': 'Quarta-feira',
    'Thursday': 'Quinta-feira',
    'Friday': 'Sexta-feira',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

dim_tempo['ano'] = dim_tempo['data'].dt.year
dim_tempo['mes'] = dim_tempo['data'].dt.month
dim_tempo['dia'] = dim_tempo['data'].dt.day
dim_tempo['trimestre'] = dim_tempo['data'].dt.quarter
dim_tempo['dia_semana'] = dim_tempo['data'].dt.day_name()
dim_tempo['dia_semana'] = dim_tempo['dia_semana'].map(dias_semana_pt)
dim_tempo['numero_dia_semana'] = dim_tempo['data'].dt.weekday
dim_tempo['eh_final_semana'] = dim_tempo['numero_dia_semana'] >= 5

dim_tempo = dim_tempo.reset_index(drop=True)
dim_tempo['sk_tempo'] = dim_tempo.index + 1

dim_tempo.head()

Dimensão 2 Local

In [ ]:
dim_local = df_concat[['br', 'municipio', 'uf', 'tipo_pista']].copy()

# Padronização
dim_local['municipio'] = dim_local['municipio'].str.strip().str.title()
dim_local['uf'] = dim_local['uf'].str.upper()
dim_local['tipo_pista'] = dim_local['tipo_pista'].str.title()

# Remove duplicados
dim_local = dim_local.drop_duplicates().reset_index(drop=True)

# Criar Surrogate Key
dim_local['sk_local'] = dim_local.index + 1
dim_local = dim_local[['sk_local', 'br', 'municipio', 'uf', 'tipo_pista']]

#SCD Tipo 2 para permanecer o histórico
dim_local['data_inicio'] = pd.Timestamp('2007-01-01')
#Aqui o atributo data_fim é para registrar se houve alguma mudança nessa rota. Exemplo essa rota passou de Rural para Urbano ou se ela está desativada.
dim_local['data_fim'] = pd.NaT
dim_local['ativo'] = True

len(dim_local)
dim_local['sk_local'].is_unique

dim_local.head()

Dimensão 3 Tipo acidente

In [ ]:
dim_tipo = df_concat[['tipo_acidente']].copy()

#Padronização
dim_tipo['tipo_acidente'] = dim_tipo['tipo_acidente'].str.strip().str.title()
#Remover Duplicados
dim_tipo = dim_tipo.drop_duplicates().reset_index(drop=True)

#Surrogate Keys
dim_tipo['sk_tipo'] = dim_tipo.index + 1

dim_tipo = dim_tipo[['sk_tipo', 'tipo_acidente']]


len(dim_tipo)
dim_tipo['sk_tipo'].is_unique
dim_tipo.isnull().sum()
dim_tipo.head()

Dimensão 4 Causa do acidente

In [ ]:
dim_causa = df_concat[['causa_acidente']].copy()

#Padronização
dim_causa['causa_acidente'] = dim_causa['causa_acidente'].str.strip().str.title()

#Remoção de duplicatas
dim_causa = dim_causa.drop_duplicates().reset_index(drop=True)

#Criação de surrogate keys:
dim_causa['sk_causa'] = dim_causa.index + 1

# Organizar colunas
dim_causa = dim_causa[['sk_causa', 'causa_acidente']]

dim_causa.head()

len(dim_causa)
dim_causa['sk_causa'].is_unique
dim_causa.isnull().sum()

Tabela fato

In [ ]:

#Padronizar as datas na tabela fato
df_concat['data'] = pd.to_datetime(
    df_concat['data_inversa'],
    format='%d/%m/%Y'
)

fato = df_concat.copy()

fato = fato.merge(
    dim_tempo[['sk_tempo', 'data']],
    on='data',
    how='left'
)

fato = fato.merge(
    dim_local[['sk_local', 'br', 'municipio', 'uf', 'tipo_pista']],
    on=['br', 'municipio', 'uf', 'tipo_pista'],
    how='left'
)

fato = fato.merge(
    dim_tipo,
    on='tipo_acidente',
    how='left'
)

fato = fato.merge(
    dim_causa,
    on='causa_acidente',
    how='left'
)

fato_acidente = fato[[
    'sk_tempo',
    'sk_local',
    'sk_tipo',
    'sk_causa',
    'mortos',
    'feridos',
    'veiculos'
]].copy()

fato_acidente.isnull().sum()

In [ ]:
fato_acidente = fato_acidente.fillna(0)
fato.columns

ETL - PASSO 3

Extract

In [ ]:
metadata = {
    "fonte": "DataTran - PRF",
    "periodo": "2007-2009",
    "data_coleta": "02/03/2026",
    "formato": "CSV"
}

In [ ]:
dim_tempo.to_sql("dim_tempo", engine, if_exists="replace", index=False)
dim_local.to_sql("dim_local", engine, if_exists="replace", index=False)
dim_tipo.to_sql("dim_tipo", engine, if_exists="replace", index=False)
dim_causa.to_sql("dim_causa", engine, if_exists="replace", index=False)
fato_acidente.to_sql("fato_acidente", engine, if_exists="replace", index=False)

Consulta

In [ ]:
conn = sqlite3.connect("dw_acidentes.db")

query = """
SELECT COUNT(*) AS total_acidentes
FROM fato_acidente
"""

df = pd.read_sql(query, conn)
df

SLICE

In [ ]:
query = """
SELECT COUNT(*) 
FROM fato_acidente f
JOIN dim_tempo t ON f.sk_tempo = t.sk_tempo
WHERE t.ano = 2008;
"""

df = pd.read_sql(query, conn)
df

DICE

In [ ]:
query = """
SELECT COUNT(*) FROM fato_acidente f
JOIN dim_tempo t ON f.sk_tempo = t.sk_tempo
JOIN dim_local l ON f.sk_local = l.sk_local
"""

df = pd.read_sql(query, conn)
df

Drill Down

In [ ]:
query = """SELECT t.ano, t.mes, COUNT(*)
FROM fato_acidente f
JOIN dim_tempo t ON f.sk_tempo = t.sk_tempo
GROUP BY t.ano, t.mes
ORDER BY t.ano, t.mes;"""

df = pd.read_sql(query, conn)
df

Roll Up

In [ ]:
query = """
SELECT l.uf, COUNT(*)
FROM fato_acidente f
JOIN dim_local l ON f.sk_local = l.sk_local
GROUP BY l.uf;
"""

df = pd.read_sql(query, conn)
df